# GEMS P6 — fast track to the leaderboard

Goal: the strongest **valid** submission before 2026-12-03 under 12 T4 GPU-h / 20 CPU-h. The B0+B1 fallback
`m9_fallback_fusion_thin.tif` stays the champion until a candidate passes its gate. Guide: `docs/p6/P6_FAST_TRACK.md`.

| section | workstream | device | exit gate |
|---|---|---|---|
| 1 | W0 leaderboard probes A-D | CPU | which evidence wins: broad discovery / near-known / budget |
| 2 | W1 seeds + 180° TTA | GPU (train, TTA) + CPU | disc ≥ fallback + 0.005, geo and worst zone no regression |
| 3 | W2 structural stacker | CPU | disc ≥ 0.167, worst ≥ 0.126, geo ≥ ref − 0.005 |
| 4 | W3 M9 throughput | GPU | ≤ 90 s / epoch, else stop M9 |
| 5 | W4 DEM pilot | CPU | ≥ 30 % trace recall @ 40 km/100 km², ≥ 3 new systems, alignment |
| 6 | W5 near-known completion | CPU | only if W0 says `near_known_completion` |
| 7 | W6 final selection + GPKG | CPU | ordered criteria |

**Rules.** Nothing is uploaded by code. Every section reads only persisted outputs, so after a disconnect run section 0,
then any section. Each workstream writes its wall time to `manifests/compute_ledger.csv` and refuses to start when its
budget is spent. Existing M9 files and checkpoints are never overwritten.

## 0. Configuration, Drive and context

Same workflow as the M9 / G02 notebooks. Prerequisites on Drive: M9 section 01 (`baseline/baseline_selection.json`,
`submissions/m9_fallback_fusion_thin.tif`) and, for W2's structural features, a G02 run.
Use a **GPU runtime** only for W1 (training / TTA) and W3; everything else runs on CPU.

In [1]:
CONFIG = {                                         # master-pipeline context (data stages are only validated / staged)
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'p6-fast-track',                          # branch or commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False, 'train_baselines': [], 'run_full_inference': False, 'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': True,                                 # needed for W1 training / TTA and W3 only
    'seed': 2026,
}
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}   # W4: download only after approval
PRIVATE_REPO = False

M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}   # opens the M9 context; no M9 section is run here
P6_CONFIG = {'overwrite_existing': False}                        # overrides of configs/p6/fast_track.yaml

In [2]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed:\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('code @', HEAD)
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('Code changed after gems_bench was imported: Runtime -> Restart session, then run from the top.')

Mounted at /content/drive
code @ 674e1405f4f8404ae3d5b1d7da3d748d22aee252


In [3]:
import json, pathlib
import numpy as np, pandas as pd
from IPython.display import display
from gems_bench.pipeline import Pipeline
from gems_bench.p6 import P6FastTrack

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for s in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(s)                                     # validated on Drive and staged locally (never regenerated)

m9 = pipe.m9(M9_CONFIG)
m9.load()
p6 = P6FastTrack(m9, P6_CONFIG)
print('P6 outputs ->', p6.ws.root)

def tab(sub, name, n=40):
    t = p6.ws.load_table(name, sub)
    display(t.head(n)) if len(t) else print('empty / not produced:', f'{sub}/{name}')
    return t

display(p6.budget.table())

[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/config_snapshot.json', 'reports/pipeline/environment_report.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/bootstrap.json']
[bootstrap] DONE: bootstrap: ok (3 checks)
[bootstrap] outputs: ['/content/drive/MyDrive/GEMS_2/environment/pipeline/config_snapshot.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/environment_report.csv']
[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['data/raw/acquisition_manifest.json', 'reports/pipeline/raw_data_acquisition.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/raw_data.json']
[raw_data] DONE: raw_data: ok (3 checks)
[raw_data] outputs: ['/content/drive/MyDrive/GEMS_2/data/raw/acquisition_manifest.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/raw_data_acquisition.csv']
[sync in] 1 copied (0.0 MiB), 0 unchanged, 1s: ['environment/pipeline/native_grid.json']
[native_grid] SKIP_CACHED: native_g

,workstream,device,cap_h,spent_h,left_h
0,w0,gpu,0.0,0.0,0.0
1,w0,cpu,0.5,0.0,0.5
2,w1,gpu,4.0,0.0,4.0
3,w1,cpu,1.5,0.0,1.5
4,w2,gpu,0.0,0.0,0.0
5,w2,cpu,6.0,0.0,6.0
6,w3,gpu,2.0,0.0,2.0
7,w3,cpu,0.5,0.0,0.5
8,w4,gpu,0.0,0.0,0.0
9,w4,cpu,6.0,0.0,6.0


## 1. W0 — leaderboard probes (no GPU)

* **A** the existing fallback file (read-only), **B** A's rule and pixel count + 300 m suppression ring,
  **C** A with the `disc_official` nested area fraction (skipped if equal to A's), **D** diagnostic: every
  template-finite non-known pixel = 1.
* Ring = *suppression*. B > A: near-known predictions are net false positives (broad discovery). B < A: public new
  truth lies next to known traces (W5). One delta is evidence about the public subset only.
* D brackets the public new-fault density (see `gems_bench/p6/probes.py`); it is never a candidate.
* **Upload manually** (3 scored submissions per rolling 7 days): A, B, C this week; D only if a slot is left.

In [4]:
log = p6.w0_probes()
display(log)
tab('probes', 'probe_plan')
print('files to upload:', p6.ws.root / 'submissions', '(A stays in the M9 submissions folder)')

[sync in] 1 copied (0.0 MiB), 0 unchanged, 0s: ['outputs/m9_geofault_production/baseline/baseline_selection.json']
[sync in] 2 copied (0.8 MiB), 0 unchanged, 1s: ['outputs/m9_geofault_production/submissions/m9_fallback_fusion_thin.tif', 'outputs/m9_geofault_production/submissions/m9_fallback_fusion_thin.json']
[sync in] 7 copied (48.7 MiB), 0 unchanged, 5s: ['reports/gems3/predictions']


/content/GEMS/gems_bench/p6/probes.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([old.reindex(columns=LOG_COLUMNS), keep], ignore_index=True) if len(keep) else \


[sync out] 11 copied (1.6 MiB), 0 unchanged, 1s: ['outputs/p6_fast_track']


,probe,submission_file,sha256,model_source,area_frac_eligible,representation,ring_px,valid_px,positive_px,positive_frac_valid,date_utc,public_score,offline_disc_dti,offline_geo_dti,notes
0,A,m9_fallback_fusion_thin.tif,a270672d66e08bc55aaf7cb5d888ec207cefcf386fc732...,s01_fusion_thin,0.050000,crest_1px,0,5167373,255171,0.049381,2026-10-27T16:34:02Z,NaN,0.161576,0.182832,existing fallback (read-only)
1,B,p6_probe_B_fusion_thin_r3.tif,00ee09f5b0191043b9ab29f9adcb4f34f796545c1856ec...,s01_fusion_thin,0.054064,crest_1px,3,5167373,255172,0.049381,2026-10-27T16:34:02Z,NaN,0.161576,NaN,A's rule and pixel count (255171 px) + 3 px su...
2,C,p6_probe_C_fusion_thin_f0.015.tif,af75a087c6785c4aa8e00f085e685fead6b94f48cd6df2...,s01_fusion_thin,0.015000,crest_1px,0,5167373,76552,0.014814,2026-10-27T16:34:03Z,NaN,NaN,NaN,disc_official nested all-zone fraction 0.015 (...
3,D,p6_probe_D_all_eligible.tif,c83063efc4d8f2e4b02b71810cd98a871f4b44045ee7da...,diagnostic_all_eligible,1.000000,all_eligible,0,5167373,5106385,0.988197,2026-10-27T16:34:03Z,NaN,NaN,NaN,DIAGNOSTIC: all template-finite non-known pixe...


,probe,ring_px,frac,match,question
0,A,0.0,0.050,NaN,real public score of the fallback
1,B,NaN,NaN,A,does removing predictions within the ring help...
2,C,0.0,0.015,NaN,is A's budget 0.05 right? C uses the disc_offi...
3,D,0.0,1.000,NaN,diagnostic bracket of the public new-fault den...


files to upload: /content/gems_work/outputs/p6_fast_track/submissions (A stays in the M9 submissions folder)


### 1b. Record public scores, then interpret

Type the public score of each uploaded file below (matched by the first characters of its SHA256), run the cell, then
read the interpretation. Missing scores stay PENDING; nothing is imputed. Copy the updated
`outputs/p6_fast_track/probes/leaderboard_log.csv` into the repository's `reports/p6/leaderboard_log.csv` when you commit.

In [5]:
SCORES = {}            # {'<sha256 prefix>': 0.1234, ...}  public leaderboard scores, typed by hand

from gems_bench.p6 import probes
L = probes.read_log(p6)
for pre, v in SCORES.items():
    hit = L.sha256.astype(str).str.startswith(pre)
    assert hit.sum() == 1, f'{pre}: {hit.sum()} matches'
    L.loc[hit, 'public_score'] = float(v)
probes.update_logs(p6, L)
display(probes.read_log(p6))
res = p6.w0_interpret()
print('VERDICT:', res['verdict'])
for f in res['findings']:
    print(' -', f)
print('missing:', res['missing_scores'], '|', res['caveat'])

,probe,submission_file,sha256,model_source,area_frac_eligible,representation,ring_px,valid_px,positive_px,positive_frac_valid,date_utc,public_score,offline_disc_dti,offline_geo_dti,notes
0,A,m9_fallback_fusion_thin.tif,a270672d66e08bc55aaf7cb5d888ec207cefcf386fc732...,s01_fusion_thin,0.050000,crest_1px,0,5167373,255171,0.049381,2026-10-27T16:34:02Z,NaN,0.161576,0.182832,existing fallback (read-only)
1,B,p6_probe_B_fusion_thin_r3.tif,00ee09f5b0191043b9ab29f9adcb4f34f796545c1856ec...,s01_fusion_thin,0.054064,crest_1px,3,5167373,255172,0.049381,2026-10-27T16:34:02Z,NaN,0.161576,NaN,A's rule and pixel count (255171 px) + 3 px su...
2,C,p6_probe_C_fusion_thin_f0.015.tif,af75a087c6785c4aa8e00f085e685fead6b94f48cd6df2...,s01_fusion_thin,0.015000,crest_1px,0,5167373,76552,0.014814,2026-10-27T16:34:03Z,NaN,NaN,NaN,disc_official nested all-zone fraction 0.015 (...
3,D,p6_probe_D_all_eligible.tif,c83063efc4d8f2e4b02b71810cd98a871f4b44045ee7da...,diagnostic_all_eligible,1.000000,all_eligible,0,5167373,5106385,0.988197,2026-10-27T16:34:03Z,NaN,NaN,NaN,DIAGNOSTIC: all template-finite non-known pixe...


VERDICT: PENDING
missing: ['A', 'B', 'C'] | public-subset evidence from single submissions; not proof of the hidden-test distribution


## 2. W1 — seeds + 180° TTA (≤ 4 GPU-h)

`seeds.plan` lists existing members and the extra seeds that fit the remaining W1 GPU budget (measured run cost when
available). Set `TRAIN_SEEDS = True` to train them (resumable experiments). TTA maps are re-predicted by the same
held-out checkpoint (`cv_fold<k>` / `disc<d>`), so they stay honest. With one member per family and no TTA, the
`seeds` variant must reproduce the fallback's numbers: check that first.

In [6]:
from pathlib import Path
import pandas as pd
import shutil

# Find existing selected_registry artifacts
SEARCH_ROOTS = [
    Path("/content/gems_work"),
    Path("/content/drive/MyDrive/GEMS_2"),
]

candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for pattern in [
        "**/features/selected_registry.parquet",
        "**/features/selected_registry.csv",
        "**/features/selected_registry.pkl",
    ]:
        candidates.extend(root.glob(pattern))

print(f"Found {len(candidates)} registry candidates")

for i, p in enumerate(candidates):
    print(f"[{i}] {p}")

if not candidates:
    raise FileNotFoundError(
        "No cached selected_registry found. "
        "M9 section 02 may never have been persisted."
    )

# Inspect target locations before copying
print("\nM9 context:", getattr(p6, "m9", None))

Found 2 registry candidates
[0] /content/drive/MyDrive/GEMS_2/outputs/m9_geofault_production/features/selected_registry.parquet
[1] /content/drive/MyDrive/GEMS_2/outputs/m9_geofault_production/features/selected_registry.csv

M9 context: <gems_bench.m9.run.M9Research object at 0x795805cefa10>


In [8]:
from pathlib import Path
from gems_bench.m9.features import selected_registry, channel_groups

m9 = p6.m9

# 1. Restore the existing M9 registry and dependencies from Drive
p6.stage_m9([
    "features/selected_registry.parquet",
    "features/selected_registry.csv",
    "features/inventory.parquet",
    "features/inventory.csv",
    "spectral/gate1.parquet",
    "spectral/gate1.csv",
    "spectral/spectral_registry.parquet",
    "spectral/spectral_registry.csv",
])

# 2. Check expected local location
print("M9 workspace:", m9.ws.root)

local_registry = (
    Path(m9.ws.root)
    / "features"
    / "selected_registry.parquet"
)

print("Local registry exists:", local_registry.exists())

# 3. Load using the actual M9 loader
reg = selected_registry(m9)

print("Registry rows:", len(reg))
print("Selected features:", reg["selected"].astype(bool).sum())

assert len(reg) == 178, "Registry was not restored correctly"
assert reg["selected"].astype(bool).sum() == 79

# 4. Verify W3 channel groups
vcfg = m9.variant_cfg(p6.cfg["throughput"]["variant"])
groups = channel_groups(m9, vcfg)

print("\nChannel groups:")
for name, channels in groups.items():
    print(f"{name}: {len(channels)}")

print("\nW3 dependencies ready.")

[sync in] 8 copied (0.1 MiB), 0 unchanged, 2s: ['outputs/m9_geofault_production/features/selected_registry.parquet', 'outputs/m9_geofault_production/features/selected_registry.csv', 'outputs/m9_geofault_production/features/inventory.parquet', 'outputs/m9_geofault_production/features/inventory.csv', 'outputs/m9_geofault_production/spectral/gate1.parquet', 'outputs/m9_geofault_production/spectral/gate1.csv', 'outputs/m9_geofault_production/spectral/spectral_registry.parquet', 'outputs/m9_geofault_production/spectral/spectral_registry.csv']
M9 workspace: /content/gems_work/outputs/m9_geofault_production
Local registry exists: True
Registry rows: 178
Selected features: 79

Channel groups:
raw: 14
struct: 65
spectral: 12
context: 23
dem: 0

W3 dependencies ready.


In [9]:
from gems_bench.p6 import seeds
display(seeds.plan(p6))
TRAIN_SEEDS = False
w1 = p6.w1_seeds(train=TRAIN_SEEDS)
display(w1['summary'])
print(json.dumps(w1['candidate'], indent=1, default=str)[:3000])

[sync in] 33 copied (934.6 MiB), 33 unchanged, 67s: ['experiments/b0_reference__d54e938a']
[sync in] 36 copied (1037.0 MiB), 35 unchanged, 69s: ['experiments/b1_bg_weight__7755635b']


,family,stem,seed,name,est_gpu_h
0,b0,b0_reference,2027,b0_reference_s2027,1.0
1,b1,b1_bg_weight,2027,b1_bg_weight_s2027,1.0
2,b0,b0_reference,2028,b0_reference_s2028,1.0
3,b1,b1_bg_weight,2028,b1_bg_weight_s2028,1.0


[sync out] 6 copied (0.0 MiB), 9 unchanged, 0s: ['outputs/p6_fast_track']
[sync in] 0 copied (0.0 MiB), 1 unchanged, 0s: ['outputs/m9_geofault_production/baseline/baseline_selection.json']
[sync out] 5 copied (0.0 MiB), 14 unchanged, 0s: ['outputs/p6_fast_track']


""


null


## 3. W2 — structural stacker (CPU, ≤ 6 h)

Honest B0 / B1 ranks + G02-C / G02-A (+ G01 persistence when present) + potential-field gradients / tilt / strike
agreement + geometry of the *permitted* known map; LightGBM fitted leave-one-zone-out with a 2 km buffer; nested
(blend, rep, frac); unique systems vs the fallback. Provenance of every channel is asserted (`LeakageError`).
The gate decides once, with no re-tuning. The full-region file is written only on PROMOTE (it needs full-grid G02
maps; their tiles are resumable).

In [10]:
w2 = p6.w2_stack(full_region=True)
for n in ('provenance', 'crossfit_audit', 'per_zone', 'unique_systems', 'feature_importance'):
    tab('stack', n)
print(json.dumps(w2['candidate'], indent=1, default=str)[:3000])

[sync in] 0 copied (0.0 MiB), 1 unchanged, 0s: ['outputs/m9_geofault_production/baseline/baseline_selection.json']
[sync in] 333 copied (4168.4 MiB), 0 unchanged, 314s: ['cache/m9/layers', 'outputs/m9_geofault_production/research/G01_wavelet_geology', 'outputs/m9_geofault_production/research/G02_structural_geology']
[p6] W2: no G02 run (no G02 run under /content/gems_work/outputs/m9_geofault_production/research/G02_structural_geology (known: ['g02_23686a63cb'])); G02 features absent


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

[p6] W2 gate: PROMOTE  (disc 0.224640288469153, worst 0.15341439472074078, geo 0.18084266130488846, +2 systems)
[sync out] 23 copied (318.5 MiB), 17 unchanged, 3s: ['outputs/p6_fast_track']


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[sync in] 0 copied (0.0 MiB), 1 unchanged, 0s: ['outputs/m9_geofault_production/baseline/baseline_selection.json']
[sync out] 7 copied (18.5 MiB), 38 unchanged, 0s: ['outputs/p6_fast_track']


...

,unit,n_train,n_pos,n_neg,status,seconds
0,geo:0,432642,41673,390969,trained,56.2
1,geo:1,432270,38547,393723,trained,46.6
2,geo:2,428797,39774,389023,trained,50.5
3,disc_official:0,251354,9112,242242,trained,50.0
4,disc_official:1,226711,8485,218226,trained,51.3
5,disc_official:2,232805,8528,224277,trained,48.1


,zone,blend,rep,frac,dti_dev,tp_w,fp_w,fn_w,n_truth,dti_heldout,task
0,0,1.0,crest_1px,0.05,0.188948,6402.954625,88192.709348,12304.045375,18707.0,0.188963,geo
1,1,0.5,crest_1px,0.03,0.192507,4685.028654,42251.101851,17322.971346,22008.0,0.173561,geo
2,2,0.5,crest_1px,0.05,0.190941,5960.175416,80285.375922,14218.824584,20179.0,0.178489,geo
3,0,1.0,skeleton_1px,0.01,0.229152,1989.339284,4667.574140,5589.660716,7579.0,0.269027,disc_official
4,1,1.0,skeleton_1px,0.01,0.233166,1978.810906,6207.504742,5490.189094,7469.0,0.259944,disc_official
5,2,0.5,skeleton_1px,0.01,0.270434,1285.806693,11492.518184,5996.193307,7282.0,0.153414,disc_official


,task,zone,stack_systems,fallback_systems,stack_only,fallback_only,n_systems
0,disc_official,0,7,8,1,2,38
1,disc_official,1,6,7,0,1,38
2,disc_official,2,6,5,1,0,44


,feature,gain
0,cnn_b1,107901.676636
1,cnn_b0,81493.743465
2,known_dist_km,50941.753251
3,grav_hgrad,34418.918099
4,tip_dist_km,27389.485757
5,mag_hgrad,25818.456403
6,mag_tilt,17599.948928
7,tip_along_cos,13572.419721
8,mg_orient_agree,8481.336061
9,cnn_disagree,6908.448769


{
 "name": "p6_stack",
 "model_source": "p6_stack",
 "disc_dti": 0.224640288469153,
 "disc_worst": 0.15341439472074078,
 "geo_dti": 0.18084266130488846,
 "geo_worst": 0.17356055226414677,
 "unique_systems": 2,
 "params_all": {
  "geo": {
   "blend": 1.0,
   "rep": "crest_1px",
   "frac": 0.05
  },
  "disc_official": {
   "blend": 1.0,
   "rep": "skeleton_1px",
   "frac": 0.01
  }
 },
 "features": [
  "cnn_b0",
  "cnn_b1",
  "cnn_disagree",
  "mag_hgrad",
  "grav_hgrad",
  "mg_orient_agree",
  "mag_tilt",
  "known_dist_km",
  "known_density",
  "tip_dist_km",
  "tip_along_cos"
 ],
 "gate": "PROMOTE",
 "gate_reasons": [],
 "incremental_disc": 0.06306407578606255
}


## 4. W3 — M9 throughput (GPU, ≤ 2 h)

Component timings on the real M9 objects (patch read, scaling, context, corruption, batch, host to device, forward /
backward, inner validation) for memory-mapped vs RAM cube and pageable vs pinned batches. Decision:
`PERMIT_ONE_CORE` (≤ 90 s / epoch: one `core` model on limited folds) or `STOP_M9`.

In [13]:
from pathlib import Path
import json
import shutil
import numpy as np

from gems_bench.m9.features import channel_groups
from gems_bench.m9.dataset import (
    channel_order, cube_key, open_cube
)

m9 = p6.m9
vcfg = m9.variant_cfg(p6.cfg["throughput"]["variant"])
groups = channel_groups(m9, vcfg)

local, context_only = channel_order(groups)

factor = int(m9.cfg["data"]["context_factor"])
key = cube_key(local, context_only, factor)

print("Expected cube key:", key)
print("Local channels:", len(local))
print("Context-only channels:", len(context_only))

cache = Path(m9.cache)
drive_cache = Path(
    "/content/drive/MyDrive/GEMS_2/cache/m9"
)

required = [
    f"cube_{key}.json",
    f"cube_{key}.npy",
    f"pooled_{key}.npy",
]

cache.mkdir(parents=True, exist_ok=True)

# Restore the exact matching cube from Drive
for filename in required:
    src = drive_cache / filename
    dst = cache / filename

    print(filename, "Drive:", src.exists(), "Local:", dst.exists())

    if not dst.exists() and src.exists():
        shutil.copy2(src, dst)

# Validate exact cache files
missing = [name for name in required if not (cache / name).exists()]

if missing:
    print("\nExact cached cube unavailable:", missing)
    print("Available Drive cubes:")
    if drive_cache.exists():
        for p in sorted(drive_cache.glob("cube_*.json")):
            print(" ", p.name)

    raise FileNotFoundError(
        "The W3 training cube was not persisted under "
        f"the expected key {key}. "
        "Do not rebuild it blindly or substitute another cube."
    )

meta = json.loads((cache / f"cube_{key}.json").read_text())
meta["cube"] = cache / f"cube_{key}.npy"
meta["pooled"] = cache / f"pooled_{key}.npy"

expected_local = local + ["m9:footprint"]
expected_pooled = expected_local + context_only

assert meta["local"] == expected_local, "Local channel mismatch"
assert meta["pooled"] == expected_pooled, "Pooled channel mismatch"
assert int(meta["factor"]) == factor, "Context factor mismatch"

cube, pooled = open_cube(meta, in_ram=False)

print("\nCube restored successfully")
print("Cube shape:", cube.shape)
print("Pooled shape:", pooled.shape)
print("Cube dtype:", cube.dtype)

Expected cube key: 9ef4f2149217
Local channels: 91
Context-only channels: 21
cube_9ef4f2149217.json Drive: False Local: False
cube_9ef4f2149217.npy Drive: False Local: False
pooled_9ef4f2149217.npy Drive: False Local: False

Exact cached cube unavailable: ['cube_9ef4f2149217.json', 'cube_9ef4f2149217.npy', 'pooled_9ef4f2149217.npy']
Available Drive cubes:


FileNotFoundError: The W3 training cube was not persisted under the expected key 9ef4f2149217. Do not rebuild it blindly or substitute another cube.

In [11]:
w3 = p6.w3_throughput()
display(w3)
print(json.loads((p6.ws.root / 'throughput' / 'decision.json').read_text()))

[sync out] 2 copied (0.0 MiB), 44 unchanged, 0s: ['outputs/p6_fast_track']


FileNotFoundError: [Errno 2] No such file or directory: '/content/gems_work/cache/m9/layers/m9_xmod_mag_grav_tol@300m_1km.npy'

## 5. W4 — DEM pilot (CPU, ≤ 6 h)

Reuses the cached M7 DEM mosaic; only if it is absent, runs M7 sections 05 / 06 with the pilot capped at 30 tiles (set
`DEM_CONFIG['download']` only after reading M7's storage estimate). Coverage masks the gate and is never a feature.

In [ ]:
w4 = p6.w4_dem()
print(json.dumps(w4, indent=1, default=str))
tab('dem', 'systems')

## 6. W5 — near-known completion (conditional)

Runs only when the W0 verdict is `near_known_completion`; otherwise it returns BLOCKED and does nothing.

In [ ]:
w5 = p6.w5_completion()
print(json.dumps(w5, indent=1, default=str)[:3000])

## 7. W6 — final selection and lineaments

Fallback + every candidate that passed its gate; order: discovery DTI, worst zone, geographic DTI, unique systems,
public score (tie-breaker). Writes `selection/decision.json` and `vectors/final_lineaments.gpkg` (segment confidence
is a heuristic rank, not a probability). Freeze by 2026-11-25; final upload by 2026-11-30.

In [ ]:
w6 = p6.w6_select()
print(json.dumps(w6, indent=1, default=str))
tab('selection', 'candidates')
tab('selection', 'elimination_trace')
display(p6.budget.table())